In [ ]:
# Install VoyageAI lib
%pip install voyageai

In [ ]:
from dotenv import load_dotenv
from anthropic import Anthropic
import voyageai
import re

anthropic_client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
system_prompt =\
"""
You are a helpful customer support chatbot for a dental clinic called BrightSmile Dental Clinic.
Respond only from the dental_clinic_policies material that is presented to you in the prompt.
The material that is presented to you in the prompt should be the only source of truth for you.
Do not pull any information from your own training data.
If for a particular question the information is not provided in the prompt,
then you should respond by saying that you don't have the necessary information to answer that particular question in a polite and gracious manner.
"""

# 1. Give the entire document - Not ideal!

In [ ]:
with open("dental_clinic_policies.md", "r", encoding="utf-8") as f:
    dental_clinic_policies = f.read()

# 2. Chunck and Retrieve

## 1. Chunk

In [ ]:
# Chunk by section
def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

with open("./dental_clinic_policies.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)

## 2. Embeddings

In [ ]:
voyageai_client = voyageai.Client()

# Embedding Generation
def generate_embedding(text, model="voyage-4-lite", input_type="query"):
    result = voyageai_client.embed(text, model=model, input_type=input_type)

    if isinstance(text, str):
        return result.embeddings[0]
    return result.embeddings

In [35]:
# VectorIndex implementation
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError(
                "Embedding function not provided during initialization."
            )
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def add_documents(self, documents: List[Dict[str, Any]]):
        if not self._embedding_fn:
            raise ValueError(
                "Embedding function not provided during initialization."
            )

        if not isinstance(documents, list):
            raise TypeError("Documents must be a list of dictionaries.")

        if not documents:
            return

        contents = []
        for i, doc in enumerate(documents):
            if not isinstance(doc, dict):
                raise TypeError(f"Document at index {i} must be a dictionary.")
            if "content" not in doc:
                raise ValueError(
                    f"Document at index {i} must contain a 'content' key."
                )
            if not isinstance(doc["content"], str):
                raise TypeError(
                    f"Document 'content' at index {i} must be a string."
                )
            contents.append(doc["content"])

        vectors = self._embedding_fn(contents)

        for vector, document in zip(vectors, documents):
            self.add_vector(vector=vector, document=document)

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError(
                    "Embedding function not provided for string query."
                )
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError(
                "Query must be either a string or a list of numbers."
            )

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(
        self, vec1: List[float], vec2: List[float]
    ) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"

In [36]:
# Create a vector store and add each embedding to it
# Note: converted to a bulk operation to avoid rate limiting errors from VoyageAI
store = VectorIndex(embedding_fn=generate_embedding)
store.add_documents([{"content": chunk} for chunk in chunks])

In [37]:
result = store.search("What time does the clinic open ?", 2)

In [41]:
print(result)

[({'content': '3. Opening Hours\n\n**Downtown Branch:**\n- Monday to Friday: 8:00 AM â€“ 7:00 PM\n- Saturday: 9:00 AM â€“ 3:00 PM\n- Sunday: Closed\n\n**Riverside Branch:**\n- Monday to Friday: 9:00 AM â€“ 5:00 PM\n- Saturday and Sunday: Closed\n\nThe clinic is closed on all national public holidays. On the day before a public holiday, both branches close at 1:00 PM. Holiday closures are announced two weeks in advance via email and on our website.\n'}, 0.3405180327727305), ({'content': '4. Booking an Appointment\n\nAppointments can be booked online through our website, by phone, or in person at either branch reception. When booking, please have your patient ID ready if you are an existing patient. New patients will be asked to provide basic contact details and insurance information.\n\nStandard check-up appointments last 30 minutes. Hygiene cleanings last 45 minutes. Longer procedures such as root canals or crown fittings are scheduled for 60 to 90 minutes. We recommend booking routine

In [ ]:
dental_clinic_policies = ""

for chunck in result:
    dental_clinic_policies += chunck[0]["content"]
    dental_clinic_policies += "\n"

In [42]:
print(dental_clinic_policies)

3. Opening Hours

**Downtown Branch:**
- Monday to Friday: 8:00 AM â€“ 7:00 PM
- Saturday: 9:00 AM â€“ 3:00 PM
- Sunday: Closed

**Riverside Branch:**
- Monday to Friday: 9:00 AM â€“ 5:00 PM
- Saturday and Sunday: Closed

The clinic is closed on all national public holidays. On the day before a public holiday, both branches close at 1:00 PM. Holiday closures are announced two weeks in advance via email and on our website.

4. Booking an Appointment

Appointments can be booked online through our website, by phone, or in person at either branch reception. When booking, please have your patient ID ready if you are an existing patient. New patients will be asked to provide basic contact details and insurance information.

Standard check-up appointments last 30 minutes. Hygiene cleanings last 45 minutes. Longer procedures such as root canals or crown fittings are scheduled for 60 to 90 minutes. We recommend booking routine check-ups at least two weeks in advance, as availability fills quick

In [43]:
messages = [
    {
        "role": "user",
        "content": \
            """
            What time does the clinic open ?

            <dental_clinic_policies>
            {dental_clinic_policies}
            </dental_clinic_policies>
            """.format(dental_clinic_policies=dental_clinic_policies)
    }
]


In [44]:
print(messages)

[{'role': 'user', 'content': '\n            What time does the clinic open ?\n\n            <dental_clinic_policies>\n            3. Opening Hours\n\n**Downtown Branch:**\n- Monday to Friday: 8:00 AM â€“ 7:00 PM\n- Saturday: 9:00 AM â€“ 3:00 PM\n- Sunday: Closed\n\n**Riverside Branch:**\n- Monday to Friday: 9:00 AM â€“ 5:00 PM\n- Saturday and Sunday: Closed\n\nThe clinic is closed on all national public holidays. On the day before a public holiday, both branches close at 1:00 PM. Holiday closures are announced two weeks in advance via email and on our website.\n\n4. Booking an Appointment\n\nAppointments can be booked online through our website, by phone, or in person at either branch reception. When booking, please have your patient ID ready if you are an existing patient. New patients will be asked to provide basic contact details and insurance information.\n\nStandard check-up appointments last 30 minutes. Hygiene cleanings last 45 minutes. Longer procedures such as root canals or c

In [45]:
# Make a request without system prompt

response = anthropic_client.messages.create(
    model=model,
    max_tokens=100,
    messages=messages,
    temperature=1.0,
    system=system_prompt)

print(response)

Message(id='msg_019fGLZLrzbZJYhgEMeWRaTD', container=None, content=[TextBlock(citations=None, text="Hello! I'd be happy to help you with our opening hours.\n\nBrightSmile Dental Clinic has two branches with different hours:\n\n**Downtown Branch:**\n- Monday to Friday: 8:00 AM – 7:00 PM\n- Saturday: 9:00 AM – 3:00 PM\n- Sunday: Closed\n\n**Riverside Branch:**\n- Monday to Friday: 9:00 AM – 5:", type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='max_tokens', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=447, output_tokens=100, server_tool_use=None, service_tier='standard'))
